In [4]:
%pip install hepdata_lib

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 42 kB 476 kB/s             
     |████████████████████████████████| 45 kB 1.1 MB/s             
     |████████████████████████████████| 98 kB 344 kB/s             
     |████████████████████████████████| 1.1 MB 10.2 MB/s            
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os, csv
import requests

from hepdata_lib import Submission, Table, Variable, Uncertainty, RootFileReader
import numpy as np

In [4]:
from pyHepMC3 import HepMC3 as hm3
import pyNuHepMC as nhm
import pyProSelecta as ps

## Original NEUT $p_{\mu}$ and $cos\theta_{\mu}$ Data Release

In [26]:
submission = Submission()
submission.add_table(create_xsec_p_mu())
submission.add_table(create_xsec_cos_theta_mu())
submission.add_additional_resource(description="Selection and projections",
    location="neutProjSelect.py",
    copy_file=True,
    file_type="ProSelecta")

submission.add_link(description="official data release", location="https://journals.aps.org/prd/abstract/10.1103/xfs2-94m3")
submission.add_link(description="Use with NUISANCE3", location="https://github.com/NUISANCEMC/nuisance3")
submission.add_link(description="Adheres to the NUISANCE HEPData Conventions", location="https://github.com/NUISANCEMC/HEPData/tree/main")
inspireID ="2940837"
ref = "PRD.112.072007"

submission.create_files(f"submission-{inspireID}", remove_old=True)

[3.09258953 3.12827428 2.72091712 3.25424338 2.99862802 3.0112705
 2.56161277 2.7416382  2.5071418  2.36059738 2.12852531 1.85854513
 1.39316546 0.46726973]
[3.0925895298277135, 3.1282742846496054, 2.720917124794506, 3.254243383645421, 2.998628019611636, 3.0112704959867025, 2.561612773234862, 2.741638196407396, 2.5071417989415754, 2.3605973820200683, 2.1285253111015616, 1.8585451299336264, 1.3931654603815011, 0.4672697293855017]
[ 6.79064 16.6115  22.4751  20.9581  27.3466  22.6254  27.8422  27.034
 25.9816  25.3157  20.8197  16.0929   8.68087  1.80263]
[0.38923001 0.42764354 0.54730522 0.63186233 0.56917221 0.57180853
 0.47579723 0.5835015  0.81345989 1.10038493 1.01058053 1.07979581
 1.05834021 1.08832164 1.15124368 1.24976918 1.51223708 1.75444664
 2.59970248 2.80475364 2.88204979 2.96559758 5.14303412 4.57922821
 4.55025142 4.65314603 4.87133565 5.77585656 5.92655051]
[0.38923000912057126, 0.42764354315247183, 0.5473052164925892, 0.631862326776965, 0.5691722059271693, 0.57180853438

In [20]:
def create_xsec_p_mu():
    data = np.loadtxt("data_1d_pmu.txt")
    data_bins = np.loadtxt("bins_1d_pmu.txt")
    data_cov = np.loadtxt("cov_1d_pmu.txt", delimiter=",")
    data_neut = np.loadtxt("neutOrigP_mu.txt")
    data_neut_2 = np.loadtxt("neutNewP_mu.txt")

    PVar = Variable("p_mu", is_independent=True, is_binned=True, units=r"$\mathrm{GeV}/c$")
    PVar.values = np.column_stack((data_bins[:-1],data_bins[1:]))

    CrossSection = Variable("cross_section", is_independent=False, is_binned=False, 
                          units=r"$\mathrm{cm}^{2} c/\mathrm{GeV} /\mathrm{Ar}$")

    # qualify the variable type and measurement type
    CrossSection.add_qualifier("variable_type", "cross_section_measurement")
    CrossSection.add_qualifier("measurement_type", "flux_averaged_differential_cross_section")
    
    # add the target specifier and probe_flux reference qualifiers
    CrossSection.add_qualifier("selectfunc", "neutProjSelect.py:passes_selection_CC_0_Pi")
    CrossSection.add_qualifier("p_mu:prettyname", r"$p_{\mu}$")
    CrossSection.add_qualifier("prettyname", r"$d\sigma/dp_{\mu}$")
    CrossSection.add_qualifier("p_mu:projectfunc", "neutProjSelect.py:projection_p_mu")

     # add the target specifier and probe_flux reference qualifiers
    CrossSection.add_qualifier("target", "Ar")
    CrossSection.add_qualifier("probe_flux", "") #absolutely no clue
    CrossSection.values = data

    # if the publication includes predictions, it is often useful to also include
    CrossSectionNEUT = Variable("cross_section_neut-prediction", is_independent=False, is_binned=False, units="$cm${}^{2} c/GeV /Ar$")
    CrossSectionNEUT.values = data_neut
    CrossSectionNEUT.add_qualifier("variable_type", "cross_section_prediction")

    #Cross section prediction from second NEUT model, more similar to the one used in the paper
    CrossSectionNEUT2 = Variable("cross_section_neut-prediction_two", is_independent=False, is_binned=False, units="$cm${}^{2} c/GeV /Ar$")
    CrossSectionNEUT2.values = data_neut_2
    CrossSectionNEUT2.add_qualifier("variable_type", "cross_section_prediction")

    TotalUncertainty = Uncertainty("total", is_symmetric=True)
    print(np.sqrt(np.diagonal(data_cov)))
    TotalUncertainty.values = np.sqrt(np.diagonal(data_cov))
    print(TotalUncertainty.values)

    CrossSection.add_uncertainty(TotalUncertainty)

    xsTable = Table("cross_section_p_mu")
    xsTable.description = """Extracted MicroBooNE cross section as a function of muon momentum compared to the nominal NEUT MC prediction."""
    xsTable.location = "FIG. 5. in the publication"
    
    xsTable.add_variable(PVar)
    xsTable.add_variable(CrossSection)
    xsTable.add_variable(CrossSectionNEUT)
    xsTable.add_variable(CrossSectionNEUT2)
    #xsTable.add_image("fig21.png")
    
    xsTable.keywords["observables"] = ["DSIG/DP"]
    xsTable.keywords["reactions"] = ["NUMU C --> MU- P"]
    xsTable.keywords["phrases"] = ["Neutrino CC0Pi", "Cross Section"]
    print(data)
    return xsTable

In [21]:
create_xsec_p_mu()

[3.09258953 3.12827428 2.72091712 3.25424338 2.99862802 3.0112705
 2.56161277 2.7416382  2.5071418  2.36059738 2.12852531 1.85854513
 1.39316546 0.46726973]
[3.0925895298277135, 3.1282742846496054, 2.720917124794506, 3.254243383645421, 2.998628019611636, 3.0112704959867025, 2.561612773234862, 2.741638196407396, 2.5071417989415754, 2.3605973820200683, 2.1285253111015616, 1.8585451299336264, 1.3931654603815011, 0.4672697293855017]
[ 6.79064 16.6115  22.4751  20.9581  27.3466  22.6254  27.8422  27.034
 25.9816  25.3157  20.8197  16.0929   8.68087  1.80263]


In [25]:
def create_xsec_cos_theta_mu():
    data = np.loadtxt("data_1d_costhetamu.txt")
    data_bins = np.loadtxt("bins_1d_ctmu.txt")
    data_cov = np.loadtxt("cov_1d_costhetamu.txt", delimiter=",")
    data_neut = np.loadtxt("neutOrig_costhetamu.txt")
    data_neut_2 = np.loadtxt("neutNew_cos_theta_mu.txt")

    CosThetaVar = Variable("cos_theta_mu", is_independent=True, is_binned=True, units="")
    CosThetaVar.values = np.column_stack((data_bins[:-1],data_bins[1:]))

    CrossSection = Variable("cross_section", is_independent=False, is_binned=False, 
                          units=r"$\mathrm{cm}^{2} c/\mathrm{GeV} /\mathrm{Ar}$")

    # qualify the variable type and measurement type
    CrossSection.add_qualifier("variable_type", "cross_section_measurement")
    CrossSection.add_qualifier("measurement_type", "flux_averaged_differential_cross_section")
    
    # add the target specifier and probe_flux reference qualifiers
    CrossSection.add_qualifier("selectfunc", "neutProjSelect.py:passes_selection_CC_0_Pi")
    CrossSection.add_qualifier("cos_theta_mu:prettyname", r"$cos\theta_{\mu}$")
    CrossSection.add_qualifier("prettyname", r"$d\sigma/dcos\theta_{\mu}$")
    CrossSection.add_qualifier("cos_theta_mu:projectfunc", "neutProjSelect.py:projection_cos_theta_mu")

     # add the target specifier and probe_flux reference qualifiers
    CrossSection.add_qualifier("target", "Ar")
    CrossSection.add_qualifier("probe_flux", "flux-offaxis-postfit-fine") #absolutely no clue
    CrossSection.values = data

    # if the publication includes predictions, it is often useful to also include
    CrossSectionNEUT = Variable("cross_section_neut-prediction", is_independent=False, is_binned=False, units="$cm${}^{2} c/GeV /Ar$")
    CrossSectionNEUT.values = data_neut
    CrossSectionNEUT.add_qualifier("variable_type", "cross_section_prediction")

    #Cross section prediction from second NEUT model, more similar to the one used in the paper
    CrossSectionNEUT2 = Variable("cross_section_neut-prediction_two", is_independent=False, is_binned=False, units="$cm${}^{2} c/GeV /Ar$")
    CrossSectionNEUT2.values = data_neut_2
    CrossSectionNEUT2.add_qualifier("variable_type", "cross_section_prediction")

    TotalUncertainty = Uncertainty("total", is_symmetric=True)
    print(np.sqrt(np.diagonal(data_cov)))
    TotalUncertainty.values = np.sqrt(np.diagonal(data_cov))
    print(TotalUncertainty.values)

    CrossSection.add_uncertainty(TotalUncertainty)

    xsTable = Table("cross_section_cos_theta_mu")
    xsTable.description = """Extracted MicroBooNE cross section as a function of muon angle compared to the nominal NEUT MC prediction."""
    xsTable.location = "FIG. 5. in the publication"
    
    xsTable.add_variable(CosThetaVar)
    xsTable.add_variable(CrossSection)
    xsTable.add_variable(CrossSectionNEUT)
    xsTable.add_variable(CrossSectionNEUT2)
    #xsTable.add_image("fig21.png")
    
    xsTable.keywords["observables"] = ["DSIG/DP"]
    xsTable.keywords["reactions"] = ["NUMU C --> MU- P"]
    xsTable.keywords["phrases"] = ["Neutrino CC0Pi", "Cross Section"]
    print(data)
    return xsTable

In [45]:
create_xsec_cos_theta_mu()

[0.38923001 0.42764354 0.54730522 0.63186233 0.56917221 0.57180853
 0.47579723 0.5835015  0.81345989 1.10038493 1.01058053 1.07979581
 1.05834021 1.08832164 1.15124368 1.24976918 1.51223708 1.75444664
 2.59970248 2.80475364 2.88204979 2.96559758 5.14303412 4.57922821
 4.55025142 4.65314603 4.87133565 5.77585656 5.92655051]
[0.38923000912057126, 0.42764354315247183, 0.5473052164925892, 0.631862326776965, 0.5691722059271693, 0.5718085343889159, 0.47579722571700644, 0.583501499569624, 0.8134598945246164, 1.1003849326485708, 1.0105805262323235, 1.0797958140315234, 1.0583402099514125, 1.0883216436329841, 1.1512436753355042, 1.249769178688609, 1.5122370845869373, 1.7544466364070468, 2.5997024829776194, 2.8047536433704834, 2.882049791381127, 2.9655975789037865, 5.143034123938903, 4.579228210080821, 4.550251421624964, 4.653146032524662, 4.871335648464393, 5.775856559853265, 5.926550514422365]
[ 2.155555  2.399303  2.869746  3.407701  3.062724  3.585309  3.398238
  4.420506  5.523495  5.010374 

## Other MicroBooNE paper

In [37]:
submission_other_paper = Submission()
submission_other_paper.add_table(create_xsec_e_cal())
submission_other_paper.add_additional_resource(description="Selection and projections",
    location="neutProjSelect.py",
    copy_file=True,
    file_type="ProSelecta")

submission_other_paper.add_link(description="official data release", location="https://doi.org/10.1103/PhysRevD.108.053002")
submission_other_paper.add_link(description="Use with NUISANCE3", location="https://github.com/NUISANCEMC/nuisance3")
submission_other_paper.add_link(description="Adheres to the NUISANCE HEPData Conventions", location="https://github.com/NUISANCEMC/HEPData/tree/main")
inspireID_other ="2621893"
ref_other = "PRD.108.053002"

submission_other_paper.create_files(f"submission-{inspireID_other}", remove_old=True)

[ 2.4285916  8.5152669 13.117739  14.625952  13.511368  11.082323
  7.7674502  4.8497867  1.8231661]


In [35]:
def create_xsec_e_cal():
    data = np.loadtxt("e_values.txt")
    data_bins = np.loadtxt("e_bin_edges.txt")
    data_unc = np.loadtxt("e_uncertainties.txt", delimiter=",")
    data_neut = np.loadtxt("neutOrig_newSignal_Ecal.txt")

    ECalVar = Variable("E_cal", is_independent=True, is_binned=True, units="")
    ECalVar.values = np.column_stack((data_bins[:-1],data_bins[1:]))

    CrossSection = Variable("cross_section", is_independent=False, is_binned=False, 
                          units=r"$\mathrm{cm}^{2} \mathrm{GeV}^2 /\mathrm{Ar}$")

    # qualify the variable type and measurement type
    CrossSection.add_qualifier("variable_type", "cross_section_measurement")
    CrossSection.add_qualifier("measurement_type", "flux_averaged_differential_cross_section")
    
    # add the target specifier and probe_flux reference qualifiers
    CrossSection.add_qualifier("selectfunc", "neutProjSelect.py:passes_selection_CC_1_p_0_Pi")
    CrossSection.add_qualifier("E_cal:prettyname", r"$cos\E_{cal}$")
    CrossSection.add_qualifier("prettyname", r"$d\sigma/dcos\E_{cal}$")
    CrossSection.add_qualifier("cos_theta_mu:projectfunc", "neutProjSelect.py:projection_e_cal")

     # add the target specifier and probe_flux reference qualifiers
    CrossSection.add_qualifier("target", "Ar")
    CrossSection.add_qualifier("probe_flux", "flux-offaxis-postfit-fine") #absolutely no clue
    CrossSection.values = data

    # if the publication includes predictions, it is often useful to also include
    CrossSectionNEUT = Variable("cross_section_neut-prediction", is_independent=False, is_binned=False, units="$cm${}^{2} /GeV^2 /Ar$")
    CrossSectionNEUT.values = data_neut
    CrossSectionNEUT.add_qualifier("variable_type", "cross_section_prediction")

    TotalUncertainty = Uncertainty("total", is_symmetric=True)

    TotalUncertainty.values = data_unc 


    CrossSection.add_uncertainty(TotalUncertainty)

    xsTable = Table("cross_section_E_cal")
    xsTable.description = """Extracted MicroBooNE cross section as a function of E_cal, the calorimetric energy reconstruction as defined at (6) in the publication compared to the nominal NEUT MC prediction."""
    xsTable.location = "FIG. 37. in the publication"
    
    xsTable.add_variable(ECalVar)
    xsTable.add_variable(CrossSection)
    xsTable.add_variable(CrossSectionNEUT)

    
    xsTable.keywords["observables"] = ["DSIG/DE_cal"]
    xsTable.keywords["reactions"] = ["NUMU C --> MU- P"]
    xsTable.keywords["phrases"] = ["Neutrino CC1p0Pi", "Cross Section"]
    print(data)
    return xsTable